In [5]:

import pandas as pd
import glob
import os

import csv


In [6]:
folder_path = r"P:\2025\NTUA GANADO LAGOON\RS\02_PRODUCTION\03_TOPODOT\XSURV_TOPO" 
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

In [7]:


def sniff_delimiter(path, default=","):
    """Try to detect delimiter; fall back to default."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        sample = f.read(4096)
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t", "|"])
        return dialect.delimiter
    except Exception:
        return default

rows = []
columns_union = set()

# First pass: read each file defensively; keep track of the union of columns
for file in csv_files:
    delimiter = sniff_delimiter(file)  # auto-detect common delimiters
    try:
        df = pd.read_csv(
            file,
            sep=delimiter,
            dtype=str,               # avoid type inference issues
            engine="python",         # more forgiving parser
            encoding="utf-8",        # try UTF-8 first
            errors="replace",        # replace bad characters
            on_bad_lines="skip",     # skip malformed rows
            quotechar='"',
            skip_blank_lines=True,
            low_memory=False,
        )
    except Exception as e:
        # Retry with Latin-1 if UTF-8 fails
        try:
            df = pd.read_csv(
                file,
                sep=delimiter,
                dtype=str,
                engine="python",
                encoding="latin-1",
                on_bad_lines="skip",
                quotechar='"',
                skip_blank_lines=True,
                low_memory=False,
            )
        except Exception as e2:
            print(f"⚠️ Skipping {os.path.basename(file)} due to error: {e2}")
            continue

    # Add a column to remember source file (helpful for tracing)
    df["__source_file"] = os.path.basename(file)
    rows.append(df)
    columns_union.update(df.columns.tolist())

# Second pass (optional): align columns so concat never fails
columns_union = list(columns_union)
aligned = []
for df in rows:
    missing = [c for c in columns_union if c not in df.columns]
    if missing:
        for c in missing:
            df[c] = pd.NA
    # Reorder consistently
    df = df[columns_union]
    aligned.append(df)

# Combine and save
if aligned:
    combined_df = pd.concat(aligned, ignore_index=True)
    out_path = os.path.join(folder_path, "combined_output.csv")
    combined_df.to_csv(out_path, index=False)
    print(f"✅ Combined {len(aligned)} files → {out_path} ({len(combined_df)} rows)")
else:
    print("❌ No valid CSV files were read.")


⚠️ Skipping US0050682.5847_251209AGW TOPO NMDOT.csv due to error: The 'low_memory' option is not supported with the 'python' engine
⚠️ Skipping US0050682.5847_251209HJF TOPO NMDOT.csv due to error: The 'low_memory' option is not supported with the 'python' engine
⚠️ Skipping US0050682.5847_251210AGW TOPO NMDOT.csv due to error: The 'low_memory' option is not supported with the 'python' engine
⚠️ Skipping US0050682.5847_251210HJF TOPO NMDOT.csv due to error: The 'low_memory' option is not supported with the 'python' engine
❌ No valid CSV files were read.
